In [0]:
%run ./00_config


In [0]:
import re
from pyspark.sql import functions as F

def to_snake(name):
    name = name.replace("\ufeff", "").strip().lower()   # remove hidden BOM character
    name = re.sub(r"[^a-z0-9]+", "_", name)
    return name.strip("_")

raw = (spark.readStream.format("cloudFiles")
       .option("cloudFiles.format", "csv")
       .option("header", "true")
       .option("cloudFiles.inferColumnTypes", "false")  .option("cloudFiles.schemaLocation", f"{CHECKPOINTS}/schema_location")  # specify schema location
        .option("cloudFiles.schemaLocation", f"{CHECKPOINTS}/schema_location")
       .option("pathGlobFilter", "*[0-9].csv")              # data file only, never MetaData
       .load(LANDING))

columns = [F.col(f"`{c}`").alias(to_snake(c)) for c in raw.columns if c != "_rescued_data"]

bronze_df = raw.select(
    *columns,
    F.col("_metadata.file_path").alias("_source_file"),
    F.current_timestamp().alias("_ingested_at"))

In [0]:
query = (bronze_df.writeStream
         .option("checkpointLocation", f"{CHECKPOINTS}/bronze")
         .trigger(availableNow=True)        # read what is new, then stop
         .toTable(f"{BRONZE}.health_indicators_raw"))
query.awaitTermination()

print("Bronze rows:", spark.table(f"{BRONZE}.health_indicators_raw").count())